# Experiment 52 - Exp22 Lambda Plateau Search

**Building on:** exp51 (OOF cmAP=0.94813), where lambda was still monotonic at 2.0 and the best TTA blend was 90/10 in favor of the normal 0s windows.

**Hypothesis:** OOF gains from increasing lambda should eventually plateau or regress. Search higher lambda values while keeping the exp51 TTA neighborhood and exp22-family weights fixed.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp49_path = nb_dir / 'exp49_exp22_public_aware.ipynb'
exp49_nb = json.loads(exp49_path.read_text())

for cell_idx in range(1, 10):
    print(f'--- executing exp49 cell {cell_idx} ---')
    src = ''.join(exp49_nb['cells'][cell_idx]['source'])
    exec(compile(src, f'exp49_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp52: lambda plateau search around exp51 best TTA settings

LAMBDAS_EXT = [2.0, 2.5, 3.0, 4.0, 5.0, 7.0]
TTA_W0S     = [0.85, 0.90, 0.95]
POWERS_EXT  = [1.1, 1.2, 1.3]
ALPHAS      = [0.08, 0.10, 0.12]
WP, WM, WPR = 0.05, 0.50, 0.45

best_cmap = 0.0
best_cfg = {}
results = []
prior_cache = {}

def get_prior_pair_exp52(lam):
    if lam not in prior_cache:
        prior_cache[lam] = (
            apply_prior(oof_raw_0, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                        prior_shrink=1.0, prior_logit_clip=None),
            apply_prior(oof_raw_250, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                        prior_shrink=1.0, prior_logit_clip=None),
        )
    return prior_cache[lam]

for lam in LAMBDAS_EXT:
    prior_0, prior_250 = get_prior_pair_exp52(lam)
    fp_0 = WP * oof_proto_0 + WM * oof_mlp_0 + WPR * prior_0
    fp_250 = WP * oof_proto_250 + WM * oof_mlp_250 + WPR * prior_250
    for tta_w0 in TTA_W0S:
        fp_avg = tta_w0 * fp_0 + (1.0 - tta_w0) * fp_250
        for power in POWERS_EXT:
            for alpha in ALPHAS:
                probs = postprocess(fp_avg, power=power, base_alpha=alpha)
                cmap = padded_cmap(Y_FULL_aligned, probs)
                row = {'lam': lam, 'tta_w0': tta_w0, 'power': power, 'alpha': alpha,
                       'wp': WP, 'wm': WM, 'wpr': WPR, 'cmap': cmap}
                results.append(row)
                if cmap > best_cmap:
                    best_cmap = cmap
                    best_cfg = row.copy()

df = pd.DataFrame(results).sort_values('cmap', ascending=False).reset_index(drop=True)
print(f'Best OOF cmAP: {best_cmap:.5f}')
print(f'Best config:   {best_cfg}')
print('\nTop 20 configs:')
print(df.head(20).to_string(index=False))
print('\nBest cmAP by lambda:')
print(df.groupby('lam')['cmap'].max().reset_index().to_string(index=False))

lam_b = best_cfg['lam']; tta_w0_b = best_cfg['tta_w0']
pow_b = best_cfg['power']; alp_b = best_cfg['alpha']
prior_0_f, prior_250_f = get_prior_pair_exp52(lam_b)
fp_0_f = WP * oof_proto_0 + WM * oof_mlp_0 + WPR * prior_0_f
fp_250_f = WP * oof_proto_250 + WM * oof_mlp_250 + WPR * prior_250_f
p_best = postprocess(tta_w0_b * fp_0_f + (1.0 - tta_w0_b) * fp_250_f,
                     power=pow_b, base_alpha=alp_b)
auc_b = macro_auc(Y_FULL_aligned, p_best)
cmap_b = padded_cmap(Y_FULL_aligned, p_best)

print('=' * 60)
print('Experiment 52 - Exp22 lambda plateau search')
print(f'Best: lambda={lam_b}, tta_w0={tta_w0_b}, power={pow_b}, alpha={alp_b}')
print(f'Weights: wp={WP}, wm={WM}, wpr={WPR}')
print(f'AUC={auc_b:.5f}  cmAP={cmap_b:.5f}')
print(f'vs exp51 OOF (0.94813): delta cmAP = {cmap_b - 0.94813:+.5f}')
print(f'vs exp22 OOF (0.93894): delta cmAP = {cmap_b - 0.93894:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
